In [1]:
import sys
sys.path.append("..")
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import load_dataset, split_data
from src.models import *
from src.evaluate import *
from src.explain import oof_shap
from src.segment import *
from sklearn.metrics import adjusted_rand_score
from src.explain import *

In [2]:
import dice_ml
from src.models import make_xgb

THRESH = {"telco": 0.36, "bank": 0.32}
TOP_SEG = {"telco": [4, 2], "bank": [2, 0]}
CONT = {"telco": ["tenure", "MonthlyCharges", "TotalCharges"],
        "bank": ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]}
VARY = {"telco": ["Contract", "PaymentMethod", "PaperlessBilling", "OnlineSecurity", "OnlineBackup",
                  "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "MonthlyCharges"],
        "bank": ["NumOfProducts", "IsActiveMember", "HasCrCard"]}

for name in ["telco", "bank"]:
    X_train, X_test, y_train, y_test = split_data(load_dataset(name))
    pipe = make_xgb(X_train).fit(X_train, y_train)
    model = ThresholdedModel(pipe, THRESH[name], X_train.dtypes)

    df = X_train.copy(); df["churn"] = y_train
    d = dice_ml.Data(dataframe=df, continuous_features=CONT[name], outcome_name="churn")
    m = dice_ml.Model(model=model, backend="sklearn")
    exp = dice_ml.Dice(d, m, method="random")

    seg = pd.read_csv(f"../results/segments_{name}.csv", index_col=0)["shap_segment"]
    proba = pd.Series(pd.read_csv(f"../results/oof_xgb_{name}.csv")["proba"].values, index=X_train.index)

    for s in TOP_SEG[name]:
        cand = proba[(seg == s) & (proba >= THRESH[name])].sort_values()
        idx = cand.index[len(cand) // 2]
        q = X_train.loc[[idx]]
        if name == "telco":
            price = float(q["MonthlyCharges"].iloc[0])
            rng = {"MonthlyCharges": [round(0.8 * price, 2), round(price, 2)]}   # at most a 20% discount
        else:
            rng = {"NumOfProducts": ["1", "2", "3", "4"]}
        cf = exp.generate_counterfactuals(q, total_CFs=3, desired_class="opposite",
                                          features_to_vary=VARY[name], permitted_range=rng)
        cfdf = cf.cf_examples_list[0].final_cfs_df.drop(columns="churn")
        cfdf.to_csv(f"../results/counterfactuals_{name}_seg{s}.csv", index=False)
        orig = q.iloc[0]
        print(f"\n=== {name} | segment {s} | customer {idx} | risk {proba[idx]:.2f} ===")
        print("original:", {c: orig[c] for c in VARY[name]})
        for i, row in cfdf.iterrows():
            print(f"  option {i + 1}:", {c: row[c] for c in VARY[name] if str(row[c]) != str(orig[c])})

100%|██████████| 1/1 [00:00<00:00,  1.84it/s]



=== telco | segment 4 | customer 5037 | risk 0.70 ===
original: {'Contract': 'Month-to-month', 'PaymentMethod': 'Mailed check', 'PaperlessBilling': 'Yes', 'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'Yes', 'StreamingMovies': 'No', 'MonthlyCharges': np.float64(79.65)}
  option 1: {'Contract': 'Two year', 'MonthlyCharges': 64.8}
  option 2: {'Contract': 'One year'}
  option 3: {'Contract': 'One year', 'MonthlyCharges': 64.18}


100%|██████████| 1/1 [00:00<00:00,  1.89it/s]



=== telco | segment 2 | customer 5750 | risk 0.51 ===
original: {'Contract': 'Month-to-month', 'PaymentMethod': 'Electronic check', 'PaperlessBilling': 'Yes', 'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'No', 'StreamingMovies': 'No', 'MonthlyCharges': np.float64(70.15)}
  option 1: {'Contract': 'Two year', 'MonthlyCharges': 64.17}
  option 2: {'Contract': 'One year', 'PaymentMethod': 'Credit card (automatic)'}
  option 3: {'Contract': 'Two year', 'PaymentMethod': 'Mailed check'}


100%|██████████| 1/1 [00:00<00:00,  2.83it/s]



=== bank | segment 2 | customer 2210 | risk 0.95 ===
original: {'NumOfProducts': np.int64(3), 'IsActiveMember': np.int64(1), 'HasCrCard': np.int64(0)}
  option 1: {'NumOfProducts': 2, 'IsActiveMember': 0}
  option 2: {'NumOfProducts': 1}
  option 3: {'NumOfProducts': 2}


100%|██████████| 1/1 [00:00<00:00,  2.27it/s]


=== bank | segment 0 | customer 5341 | risk 0.63 ===
original: {'NumOfProducts': np.int64(1), 'IsActiveMember': np.int64(0), 'HasCrCard': np.int64(0)}
  option 1: {'IsActiveMember': 1, 'HasCrCard': 1}
  option 2: {'NumOfProducts': 2, 'IsActiveMember': 1}
  option 3: {'IsActiveMember': 1, 'HasCrCard': 1}
